<a href="https://colab.research.google.com/github/corneliu-aursulesei/angular-cli/blob/master/Copie_a_blocnotesului_2_MIoTy_Analiza_Telegrame_Zona_Luna_v1_8_(1).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# MIoT -- Analiza Telegrame Contoare Apa
### Notebook complet: montare Drive -> extractie -> CSV-uri -> rata de citire
**Versiune:** v1.7 | **Protocoale:** T1, R3, L1C (MIOTY), R4, R4P, R4N

---
**Instructiuni:**
1. Ruleaza celulele **in ordine** de sus in jos
2. La Pasul 2 va aparea o fereastra de autorizare Google Drive
3. Modifica **doar** Pasul 4 (CONFIGURARE)
4. Dupa Pasul 4 poti rula restul cu Runtime > Run after


## Pasul 1 — Instalare pachete necesare

In [ ]:
# ════════════════════════════════════════════════════════════════
# PASUL 1 — Instalare pachete
# Rulează o singură dată per sesiune Colab
# ════════════════════════════════════════════════════════════════
import subprocess, sys

print('Instalare pachete...')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'pandas', 'lxml', 'openpyxl'], check=True)
print('OK: Pachete instalate: pandas, lxml, openpyxl')

Instalare pachete...
OK: Pachete instalate: pandas, lxml, openpyxl


## Pasul 2 — Conectare si montare Google Drive

In [ ]:
# ════════════════════════════════════════════════════════════════
# PASUL 2 — Montare Google Drive
# Va apărea o fereastră de autorizare — click 'Connect to Google Drive'
# și alege contul Google unde ai folderele cu date
# ════════════════════════════════════════════════════════════════
from google.colab import drive
import os

print('Montare Google Drive...')
print('   -> Va apărea o fereastră de autorizare')
print('   -> Selectează contul Google corect')
print('   -> Click "Connect to Google Drive"')
print()

drive.mount('/content/drive')

print()
print('OK: Google Drive montat cu succes!')
print(f'   Calea de acces: /content/drive/MyDrive')

# Verificare că Drive-ul e accesibil
if os.path.exists('/content/drive/MyDrive'):
    print('OK: MyDrive accesibil OK')
else:
    print('EROARE: EROARE: MyDrive nu este accesibil!')
    print('   Verifică autorizarea și reîncearcă.')

Montare Google Drive...
   -> Va apărea o fereastră de autorizare
   -> Selectează contul Google corect
   -> Click "Connect to Google Drive"

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).

OK: Google Drive montat cu succes!
   Calea de acces: /content/drive/MyDrive
OK: MyDrive accesibil OK


## Pasul 3 — Verificare structură foldere în Drive

In [ ]:
# ════════════════════════════════════════════════════════════════
# PASUL 3 — Verificare că folderele necesare există în Drive
# Această celulă listează ce găsește în Drive pentru a confirma
# că structura de foldere este corectă înainte de procesare
# ════════════════════════════════════════════════════════════════
import os
from pathlib import Path

DRIVE = '/content/drive/MyDrive'

# Foldere de verificat
check_paths = [
    f'{DRIVE}/RD_03',
    f'{DRIVE}/deva',
]

print('Verificare structură Drive:')
print()
for p in check_paths:
    if os.path.exists(p):
        contents = os.listdir(p)
        print(f'  OK: {p}')
        for item in sorted(contents)[:10]:  # primele 10
            full = os.path.join(p, item)
            tip  = '' if os.path.isdir(full) else ''
            print(f'     {tip} {item}')
        if len(contents) > 10:
            print(f'     ... și încă {len(contents)-10} elemente')
    else:
        print(f'  EROARE: NU EXISTĂ: {p}')
    print()

print('INFO: Dacă folderele sunt corecte, continuă cu Pasul 4 (CONFIGURARE).')
print('INFO: Dacă un folder lipsește, verifică structura Drive-ului.')

Verificare structură Drive:

  OK: /content/drive/MyDrive/RD_03
      06
      10
      2025_06_b
      2025_10_b

  OK: /content/drive/MyDrive/deva
      10

INFO: Dacă folderele sunt corecte, continuă cu Pasul 4 (CONFIGURARE).
INFO: Dacă un folder lipsește, verifică structura Drive-ului.


## Pasul 4 —  CONFIGURARE (modifică doar aici!)

In [ ]:
# ================================================================
# CONFIGURARE v1.7 -- MODIFICA DOAR ACEASTA CELULA
# ================================================================
import os

ZONA        = 'Z9'       # Zona: 'Z1' ... 'Z9'
LUNA        = '10'       # Luna: '10' = octombrie | '06' = iunie
LUNA_FOLDER = '2025-10'  # Folder sursa: '2025-10' | '2025-06'
AN          = '2025'     # Anul

# -- Cai derivate automat -- NU modifica --
DRIVE_ROOT  = '/content/drive/MyDrive'
SURSA_BASE  = f'{DRIVE_ROOT}/RD_03/{AN}_{LUNA}_b/{LUNA_FOLDER}'
LISTA_PATH  = f'{DRIVE_ROOT}/deva/{LUNA}/{ZONA}/lista.txt'
OUTPUT_DIR  = f'{DRIVE_ROOT}/deva/{LUNA}/{ZONA}'
ZONA_NR     = ZONA.lower().replace('z', '')
LOG_PATH    = f'{OUTPUT_DIR}/z{ZONA_NR}_{LUNA}_executie.log'

print('=== CONFIGURARE ===')
print(f'  Zona        : {ZONA}')
print(f'  Luna        : {LUNA}')
print(f'  An          : {AN}')
print(f'  Sursa       : {SURSA_BASE}')
print(f'  Lista       : {LISTA_PATH}')
print(f'  Output      : {OUTPUT_DIR}')
print()

# -- Verificare cai --
ok = True

if os.path.exists(SURSA_BASE):
    print(f'[OK]  SURSA gasita')
else:
    print(f'[ERR] SURSA lipseste: {SURSA_BASE}')
    ok = False

if os.path.exists(LISTA_PATH):
    print(f'[OK]  lista.txt gasit')
else:
    print(f'[ERR] lista.txt lipseste: {LISTA_PATH}')
    ok = False

os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f'[OK]  Folder output pregatit')
print()

if ok:
    print('>>> Configurare OK - continua cu urmatoarele celule')
else:
    print('>>> ERORI in configurare - verifica caile de mai sus!')


=== CONFIGURARE ===
  Zona        : Z9
  Luna        : 10
  An          : 2025
  Sursa       : /content/drive/MyDrive/RD_03/2025_10_b/2025-10
  Lista       : /content/drive/MyDrive/deva/10/Z9/lista.txt
  Output      : /content/drive/MyDrive/deva/10/Z9

[OK]  SURSA gasita
[OK]  lista.txt gasit
[OK]  Folder output pregatit

>>> Configurare OK - continua cu urmatoarele celule


## Pasul 5 — Import librării și funcții utilitare

In [ ]:
# ════════════════════════════════════════════════════════════════
# PASUL 5 — Import librării + funcția de logging cu timestamp
# ════════════════════════════════════════════════════════════════
import gzip, re, os, sys
from datetime import datetime
from pathlib import Path
from collections import defaultdict
import pandas as pd
import xml.etree.ElementTree as ET

def log(msg, level='INFO'):
    ts = datetime.now().strftime('%H:%M:%S')
    linie = '[' + ts + '] [' + level + '] ' + str(msg)
    print(linie)
    try:
        with open(LOG_PATH, 'a', encoding='utf-8') as lf:
            lf.write(linie)
            lf.write('\n')
    except Exception as e:
        print('[' + ts + '] [WARN] Nu pot scrie in log: ' + str(e))

# Cadența nominală per protocol (secunde între telegrame)
CADENTA = {
    'T1' : 16,
    'R3' : 16,
    'L1C': 3600,
    'R4' : 22,
    'R4P': 22,
    'R4N': 22,
}

# Grupare protocoale pentru fișiere separate
PROTO_GROUP = {
    'T1' :'T1R3', 'R3' :'T1R3',
    'L1C':'L1C',
    'R4' :'R4',   'R4P':'R4',  'R4N':'R4',
}

def get_group(proto):
    return PROTO_GROUP.get(str(proto).upper().strip(), 'OTHER')

log('Librării importate OK', 'OK')
log(f'pandas {pd.__version__}', 'INFO')
def log(msg, level='INFO'):
    """Afiseaza mesaj cu timestamp si il scrie si in fisierul de log."""
    ts = datetime.now().strftime('%H:%M:%S')
    linie = f'[{ts}] [{level}] {msg}'
    print(linie)
    try:
        with open(LOG_PATH, 'a', encoding='utf-8') as lf:
            lf.write(linie + '\n')
    except Exception as e:
        print(f'[WARN] Nu pot scrie in log: {e}')

# -- Initializare fisier log --
import os as _os
_os.makedirs(OUTPUT_DIR, exist_ok=True)
with open(LOG_PATH, 'w', encoding='utf-8') as lf:
    lf.write(f'LOG EXECUTIE -- {ZONA} / Luna {LUNA} / {AN}\n')
    lf.write(f'Notebook: MIoT_Analiza_Telegrame_Zona_Luna_v1.7\n')
    lf.write(f'Start: {datetime.now().strftime("%Y-%m-%d %H:%M:%S")}\n')
    lf.write('-' * 60 + '\n')
log(f'Log de executie initializat: {LOG_PATH}', 'INFO')
log(f'Versiune notebook: v1.7', 'INFO')

# -- Initializare fisier log --
import os as _os
_os.makedirs(OUTPUT_DIR, exist_ok=True)
with open(LOG_PATH, 'w', encoding='utf-8') as lf:
    lf.write(f'LOG EXECUTIE -- {ZONA} / Luna {LUNA} / {AN}\n')
    lf.write(f'Notebook: MIoT_Analiza_Telegrame_Zona_Luna_v1.7\n')
    lf.write(f'Start: {datetime.now().strftime("%Y-%m-%d %H:%M:%S")}\n')
    lf.write('-' * 60 + '\n')
log(f'Log initializat: {LOG_PATH}', 'INFO')
log('Versiune notebook: v1.7', 'INFO')


[08:50:18] [OK] Librării importate OK
[08:50:18] [INFO] pandas 2.2.2
[08:50:18] [INFO] Log de executie initializat: /content/drive/MyDrive/deva/10/Z9/z9_10_executie.log
[08:50:18] [INFO] Versiune notebook: v1.7
[08:50:18] [INFO] Log initializat: /content/drive/MyDrive/deva/10/Z9/z9_10_executie.log
[08:50:18] [INFO] Versiune notebook: v1.7


## Pasul 6 — Citire lista.txt și extragere contoare zonă

In [ ]:
# ================================================================
# PASUL 6 -- Citire lista.txt
# Format: tab-separated, fara header, 3 coloane:
#   col 0: mac_rdc   ex: M70B3D51F1CC4
#   col 1: zona      ex: Z6
#   col 2: sector_id ex: ABCS0001
# ================================================================
import pandas as pd

log(f'Citire lista.txt pentru zona {ZONA}...', 'INFO')
log(f'Cale: {LISTA_PATH}', 'INFO')

df_raw = pd.read_csv(
    LISTA_PATH,
    sep='\t',
    header=None,
    names=['mac_rdc', 'zona', 'sector_id'],
    dtype=str
)
log(f'Randuri citite: {len(df_raw)}', 'INFO')

# Curatare spatii
for col in df_raw.columns:
    df_raw[col] = df_raw[col].str.strip()

# Afisare continut complet pentru verificare
log('Continut lista.txt:', 'INFO')
for _, row in df_raw.iterrows():
    log(f'  mac={row["mac_rdc"]}  zona={row["zona"]}  sector={row["sector_id"]}', 'INFO')

# Filtrare zona curenta
df_zona = df_raw[df_raw['zona'].str.upper() == ZONA.upper()].copy()
log(f'Randuri pentru {ZONA}: {len(df_zona)}', 'INFO')

# Dictionar mac_rdc -> sector_id
mac_to_sector = {}
for _, row in df_zona.iterrows():
    mac = str(row['mac_rdc']).upper()
    sector = str(row['sector_id'])
    if mac and mac not in ('NAN', 'NONE', ''):
        mac_to_sector[mac] = sector

MAC_RDC_ZONE = set(mac_to_sector.keys())
log(f'MAC-uri RDC pentru {ZONA}: {len(MAC_RDC_ZONE)}', 'OK')
for mac, sector in sorted(mac_to_sector.items()):
    log(f'  {mac} -> sector: {sector}', 'INFO')

if not MAC_RDC_ZONE:
    log('NICIUN mac_rdc gasit! Verifica lista.txt.', 'ERR')


[08:50:18] [INFO] Citire lista.txt pentru zona Z9...
[08:50:18] [INFO] Cale: /content/drive/MyDrive/deva/10/Z9/lista.txt
[08:50:18] [INFO] Randuri citite: 3
[08:50:18] [INFO] Continut lista.txt:
[08:50:18] [INFO]   mac=MD83ADD08F687  zona=Z9  sector=ABBY0001a
[08:50:18] [INFO]   mac=MD83ADD08F4F5  zona=Z9  sector=ABCB0001
[08:50:18] [INFO]   mac=M70B3D51F1CD8  zona=Z9  sector=ABBY0001b
[08:50:18] [INFO] Randuri pentru Z9: 3
[08:50:18] [OK] MAC-uri RDC pentru Z9: 3
[08:50:18] [INFO]   M70B3D51F1CD8 -> sector: ABBY0001b
[08:50:18] [INFO]   MD83ADD08F4F5 -> sector: ABCB0001
[08:50:18] [INFO]   MD83ADD08F687 -> sector: ABBY0001a


## Pasul 7 — Funcții decodare serial fizic contor

In [ ]:
# ════════════════════════════════════════════════════════════════
# PASUL 7 — Funcții decodare deviceID -> serial fizic contor
#
# FORMAT MD (tag <md>): deviceId numeric, ultimele 8 cifre, little-endian
#   Ex: '00A511760727980288'
#       -> ultimele 8: '27980288'
#       -> split blocuri 2: ['27','98','02','88']
#       -> swap cifre fiecare bloc: ['72','89','20','88']
#       -> reverse lista: ['88','20','89','72']
#       -> serial: '88208972'
#
# FORMAT SDL (tag <sdl>): 'UDME760786792976000'
#       -> 8 cifre înainte de '000' final -> '86792976'
# ════════════════════════════════════════════════════════════════

def decode_md(device_id_raw):
    """Decodare format <md> tag: little-endian swap pe blocuri de 2."""
    try:
        s = str(device_id_raw).strip()
        last8 = s[-8:]
        if len(last8) < 8:
            return s, 'MD_SHORT'
        blocks  = [last8[i:i+2] for i in range(0, 8, 2)]
        swapped = [b[::-1] for b in blocks]
        serial  = ''.join(swapped[::-1])
        return serial, 'MD'
    except Exception as e:
        return str(device_id_raw), f'MD_ERR:{e}'

def decode_sdl(device_id_raw):
    """Decodare format <sdl> tag: 8 cifre înainte de '000' final."""
    try:
        s = str(device_id_raw).strip()
        m = re.search(r'(\d{8})0{3}$', s)
        if m:
            return m.group(1), 'SDL'
        # Fallback: cifre din pozițiile -11 până la -3
        digits = re.findall(r'\d', s)
        if len(digits) >= 11:
            return ''.join(digits[-11:-3]), 'SDL_FB'
        return s, 'SDL_ERR'
    except Exception as e:
        return str(device_id_raw), f'SDL_ERR:{e}'

# ── Test automat al funcțiilor ──
print('Test funcții decodare:')
print()
tests = [
    ('MD',  '00A511760727980288',   '88208972'),
    ('SDL', 'UDME760786792976000',  '86792976'),
    ('SDL', 'UDME760787299413000',  '87299413'),
    ('SDL', 'UDME760786959592000',  '86959592'),
]
for fmt, raw, expected in tests:
    if fmt == 'MD':
        result, f = decode_md(raw)
    else:
        result, f = decode_sdl(raw)
    status = 'OK' if result == expected else 'ATENTIE '
    print(f'  {status} {fmt}  {raw} -> {result}  (așteptat: {expected})')

print()
log('Funcții decodare deviceID OK', 'OK')

Test funcții decodare:

  OK MD  00A511760727980288 -> 88208972  (așteptat: 88208972)
  OK SDL  UDME760786792976000 -> 86792976  (așteptat: 86792976)
  OK SDL  UDME760787299413000 -> 87299413  (așteptat: 87299413)
  OK SDL  UDME760786959592000 -> 86959592  (așteptat: 86959592)

[08:50:18] [OK] Funcții decodare deviceID OK


## Pasul 8 — Funcție de parsare arhivă .gz XML

In [ ]:
# ================================================================
# PASUL 8 -- Functii de citire fisiere si parsare XML
#
# Formate XML suportate:
#   FORMAT MD  (gateway vechi M70...): <mdp> + <md recType="RSSI|SNR" deviceId="...">
#   FORMAT SDL (gateway nou MD8/ME4...): <mdp> + <sdl> + <sd phy="T1|L1C|R3">
#
# IMPORTANT: XML-ul foloseste namespace http://www.diehl-metering.com/
# Toate tag-urile trebuie cautate cu namespace explicit: {NS}mdp, {NS}md, {NS}sdl, {NS}sd
#
# Logica filtrare:
#   - FORMAT MD : filtram pe mdp.source (MAC gateway) -> e in MAC_RDC_ZONE
#   - FORMAT SDL: filtram pe mdp.source (MAC gateway) -> e in MAC_RDC_ZONE
#   - deviceId contor se extrage din primii 10 bytes ai frame-ului wMBUS (data field)
# ================================================================

import zlib

NS_DIEHL = 'http://www.diehl-metering.com/'

def ts_to_human(ts_str):
    """Conversie timestamp Unix -> string human-readable UTC."""
    try:
        if str(ts_str).isdigit():
            return pd.to_datetime(int(ts_str), unit='s', utc=True).strftime('%Y-%m-%d %H:%M:%S')
    except:
        pass
    return ''

def make_row(serial, raw, fmt, mac, sector, proto, ts, ms,
             nbytes, rssi, snr, qi1, qi2, qi3,
             aid, rs, re_, status, src_file):
    """Construieste un rand al bazei de date (23 coloane)."""
    return {
        'serial_fizic'    : serial,
        'device_id_raw'   : raw,
        'decode_fmt'      : fmt,
        'mac_rdc'         : mac,
        'sector_id'       : sector,
        'zona'            : ZONA,
        'protocol'        : proto,
        'protocol_group'  : get_group(proto) if proto else None,
        'ts'              : ts,
        'ts_human'        : ts_to_human(ts),
        'delta_ms'        : ms,
        'bytes'           : nbytes,
        'rssi'            : rssi,
        'snr'             : snr,
        'mioty_qi_1'      : qi1,
        'mioty_qi_2'      : qi2,
        'mioty_qi_3'      : qi3,
        'aID'             : aid,
        'record_start'    : rs,
        'record_end'      : re_,
        'rx_count_same_ts': 1,
        'status'          : status,
        'gz_source'       : src_file,
    }

def deviceid_from_data(data_hex):
    """
    Extrage deviceId (format: 00MMMMVVTTAAAAAAAA) din primii 10 bytes
    ai unui frame wMBUS (hex string).
    wMBUS layout: [0]=L [1]=C [2..3]=Mfr [4..7]=ID [8]=Ver [9]=Type
    Returneaza string uppercase sau None la eroare.
    """
    try:
        b = bytes.fromhex(data_hex[:20])
        M, A, Ver, Typ = b[2:4], b[4:8], b[8], b[9]
        return f'00{M[0]:02X}{M[1]:02X}{Ver:02X}{Typ:02X}{A[0]:02X}{A[1]:02X}{A[2]:02X}{A[3]:02X}'
    except:
        return None

def read_file_content(file_path):
    """
    Citeste continutul unui fisier indiferent de tip:
      .gz  -> dezarhiveaza gzip
      .z   -> dezarhiveaza cu zlib / gzip
      .xml -> citeste direct text
    Returneaza: (content_string, tip_detectat, eroare_sau_None)
    """
    ext = Path(file_path).suffix.lower()

    if ext == '.gz':
        try:
            with gzip.open(file_path, 'rt', encoding='utf-8', errors='replace') as f:
                return f.read(), 'gz', None
        except Exception as e:
            return None, 'gz', str(e)

    if ext == '.z':
        try:
            with open(file_path, 'rb') as f:
                raw_bytes = f.read()
            try:
                content = zlib.decompress(raw_bytes, wbits=-15).decode('utf-8', errors='replace')
                return content, 'z_deflate', None
            except:
                pass
            try:
                import io
                with gzip.open(io.BytesIO(raw_bytes), 'rt', encoding='utf-8', errors='replace') as f:
                    return f.read(), 'z_as_gz', None
            except:
                pass
            return raw_bytes.decode('utf-8', errors='replace'), 'z_raw', None
        except Exception as e:
            return None, 'z', str(e)

    try:
        with open(file_path, 'r', encoding='utf-8', errors='replace') as f:
            return f.read(), 'xml', None
    except Exception as e:
        return None, 'xml', str(e)

def parse_xml_content(content, src_file, mac_filter, mac_sector):
    """
    Parseaza continut XML si extrage telegramele contoarelor.

    Suporta doua formate Diehl (ambele cu namespace http://www.diehl-metering.com/):

    FORMAT MD (gateway vechi M70...):
      <mdp source="MAC_GW" recordStart="..." recordEnd="...">
        <md recType="RSSI|SNR" recVal="..." deviceId="00MMVVTTAAAA" time="..." raType="R3|T1|L1C">
          <ra>HEX_DATA</ra>
        </md>
      </mdp>
      Filtru: mdp.source in mac_filter (MAC gateway din zona)
      deviceId al contorului: din atribut md.deviceId direct

    FORMAT SDL (gateway nou MD8.../ME4...):
      <mdp source="MAC_GW">
        <sdl deviceId="UDME...">
          <sd phy="T1|L1C|R3" rssi="..." snr="..." ts="..." aID="...">
            <data>HEX_DATA</data>
            <metaData key="miotyQi">qi1,qi2,qi3</metaData>  (optional)
          </sd>
        </sdl>
      </mdp>
      Filtru: mdp.source in mac_filter (MAC gateway din zona)
      deviceId al contorului: extras din primii 10 bytes ai data field

    mac_filter : set de MAC-uri RDC (gateway) uppercase din lista.txt zona curenta
    mac_sector : dict MAC_RDC -> sector_id (nu se foloseste pentru deviceId contor)
    """
    NS = NS_DIEHL
    rows, silents = [], []

    # Sterge XML declaration si escapeaza & neescapatate
    content_clean = re.sub(r'<\?xml[^?]*\?>', '', content).strip()
    content_clean = re.sub(r'&(?!amp;|lt;|gt;|quot;|apos;)', '&amp;', content_clean)
    xml_str = '<root>' + content_clean + '</root>'
    try:
        root_el = ET.fromstring(xml_str)
    except ET.ParseError as e:
        return rows, silents, f'ERR_XML:{e}'

    # ================================================================
    # FORMAT MD: <mdp> + <md recType="RSSI|SNR">
    # ================================================================
    for mdp in root_el.iter(f'{{{NS}}}mdp'):
        src_mac = str(mdp.get('source', '')).strip().upper()
        if src_mac not in mac_filter:
            continue
        rs  = mdp.get('recordStart', '')
        re_ = mdp.get('recordEnd', '')
        sector = mac_sector.get(src_mac, '')

        md_grp = defaultdict(dict)
        for md in mdp.iter(f'{{{NS}}}md'):
            dev   = str(md.get('deviceId', '')).strip().upper()
            rtype = md.get('recType', '').upper()
            rval  = md.get('recVal', '')
            ts_v  = md.get('time', '')
            proto = str(md.get('raType', '')).upper()
            mod   = md.get('moduleId', '')
            ra_el = md.find(f'{{{NS}}}ra')
            ra_txt = ra_el.text.strip() if ra_el is not None and ra_el.text else ''
            nb_b  = len(ra_txt) // 2
            k = (dev, ts_v, proto)
            md_grp[k].update({'dev': dev, 'ts': ts_v, 'proto': proto,
                               'bytes': nb_b, 'mod': mod})
            if rtype == 'RSSI': md_grp[k]['rssi'] = rval
            elif rtype == 'SNR': md_grp[k]['snr']  = rval

        for k, v in md_grp.items():
            serial, fmt = decode_md(v.get('dev', ''))
            rows.append(make_row(
                serial, v.get('dev',''), fmt, src_mac, sector,
                v.get('proto',''), v.get('ts',''),
                None, v.get('bytes', 0),
                v.get('rssi', None), v.get('snr', None),
                None, None, None, v.get('mod', None),
                rs, re_, 'RECEIVED', src_file
            ))

    # ================================================================
    # FORMAT SDL: <mdp> + <sdl> + <sd phy="T1|L1C|R3">
    # ================================================================
    for mdp in root_el.iter(f'{{{NS}}}mdp'):
        src_mac = str(mdp.get('source', '')).strip().upper()
        if src_mac not in mac_filter:
            continue
        rs  = mdp.get('recordStart', '')
        re_ = mdp.get('recordEnd', '')
        sector = mac_sector.get(src_mac, '')

        for sdl in mdp.iter(f'{{{NS}}}sdl'):
            sds = sdl.findall(f'{{{NS}}}sd')

            if not sds:
                # SDL fara sd = contor silent in aceasta ora
                # Extragem deviceId din UDME format
                dev_raw = sdl.get('deviceId', '')
                serial, fmt = decode_sdl(dev_raw)
                silents.append(make_row(
                    serial, dev_raw, fmt, src_mac, sector,
                    None, rs, None, 0,
                    None, None, None, None, None, None,
                    rs, re_, 'SILENT_SDL', src_file
                ))
                continue

            for sd in sds:
                proto  = str(sd.get('phy', '')).upper().strip()
                rssi   = sd.get('rssi', None)
                snr    = sd.get('snr', None)
                ts_v   = sd.get('ts', '')
                ms     = sd.get('ms', None)
                aid    = sd.get('aID', None)
                data_el = sd.find(f'{{{NS}}}data')
                dtxt   = data_el.text.strip() if data_el is not None and data_el.text else ''
                nb_b   = len(dtxt) // 2

                # Extrage deviceId contor din data field
                dev_id = deviceid_from_data(dtxt)
                if not dev_id:
                    continue
                dev_id = dev_id.upper()

                qi1 = qi2 = qi3 = None
                meta = sd.find(f'{{{NS}}}metaData[@key="miotyQi"]')
                if meta is not None and meta.text:
                    pts = meta.text.strip().split(',')
                    try: qi1 = float(pts[0])
                    except: pass
                    try: qi2 = float(pts[1])
                    except: pass
                    try: qi3 = float(pts[2])
                    except: pass

                serial, fmt = decode_md(dev_id)
                rows.append(make_row(
                    serial, dev_id, fmt, src_mac, sector,
                    proto, ts_v, ms, nb_b,
                    rssi, snr, qi1, qi2, qi3, aid,
                    rs, re_, 'RECEIVED', src_file
                ))

    return rows, silents, 'OK'

def scan_subfolder(subfolder):
    """Detecteaza tipurile de fisiere dintr-un subfolder."""
    sf = Path(subfolder)
    gz_files  = sorted(sf.glob('*.gz'))
    z_files   = sorted(sf.glob('*.z'))
    xml_files = sorted(sf.glob('*.xml'))
    summary = []
    if gz_files:  summary.append(f'{len(gz_files)} x .gz')
    if z_files:   summary.append(f'{len(z_files)} x .z')
    if xml_files: summary.append(f'{len(xml_files)} x .xml')
    all_files = list(gz_files) + list(z_files) + list(xml_files)
    tip_str   = ', '.join(summary) if summary else 'NICIUN fisier recunoscut'
    return all_files, tip_str

log('Functii citire + parsare XML initializate OK', 'OK')
log(f'Namespace Diehl: {NS_DIEHL}', 'INFO')
log('Format MD  (M70...): <md recType=RSSI/SNR deviceId=...>', 'INFO')
log('Format SDL (MD8/ME4): <sdl>/<sd phy=T1/L1C/R3> -> deviceId din data bytes', 'INFO')


[08:50:18] [OK] Functii citire + parsare XML initializate OK
[08:50:18] [INFO] Namespace Diehl: http://www.diehl-metering.com/
[08:50:18] [INFO] Format MD  (M70...): <md recType=RSSI/SNR deviceId=...>
[08:50:18] [INFO] Format SDL (MD8/ME4): <sdl>/<sd phy=T1/L1C/R3> -> deviceId din data bytes


## Pasul 9 —  Procesare arhive .gz (poate dura câteva minute)

In [ ]:
# ================================================================
# PASUL 9 -- Procesare subfoldere gateway (doar cele din zona)
#
# Logica:
#   1. Listam toate subfolder-ele din SURSA_BASE
#   2. Numele subfolder-ului = MAC gateway (ex: MD83ADD08F3D3)
#   3. SKIP imediat daca MAC nu e in MAC_RDC_ZONE -> zero fisiere deschise
#   4. Doar subfolder-ele din lista.txt sunt dezarhivate si parsate
#
# Eficienta: daca zona are 5 gateway-uri dintr-un total de 200,
# se proceseaza doar 5 subfoldere, nu 200.
# ================================================================
log(f'START procesare {ZONA} / Luna {LUNA} / {AN}', 'INFO')
log(f'Sursa: {SURSA_BASE}', 'INFO')
log(f'Gateway-uri zona {ZONA}: {sorted(MAC_RDC_ZONE)}', 'INFO')
print()

all_dirs = sorted([d for d in Path(SURSA_BASE).iterdir() if d.is_dir()])
log(f'Total subfoldere (toate zonele): {len(all_dirs)}', 'INFO')

# Filtram imediat: doar subfolder-ele al caror nume e in MAC_RDC_ZONE
subfolders_zona = [d for d in all_dirs if d.name.upper() in MAC_RDC_ZONE]
skipped         = len(all_dirs) - len(subfolders_zona)
log(f'Subfoldere zona {ZONA}: {len(subfolders_zona)} procesate, {skipped} sarite', 'INFO')

if not subfolders_zona:
    log(f'ATENTIE: Niciun subfolder gasit pentru {ZONA}!', 'ERR')
    log(f'Verifica ca MAC-urile din lista.txt corespund cu numele folderelor.', 'ERR')
else:
    for sf in subfolders_zona:
        files, tip_str = scan_subfolder(sf)
        log(f'  {sf.name:<20} {tip_str}', 'INFO')
print()

all_rows   = []
all_silent = []
total_files = 0
total_err   = 0
total_recv  = 0
total_sil   = 0
t_start = datetime.now()

for sf_idx, subfolder in enumerate(subfolders_zona, 1):
    mac_gw = subfolder.name.upper()
    files, tip_str = scan_subfolder(subfolder)

    if not files:
        log(f'[{sf_idx}/{len(subfolders_zona)}] {subfolder.name} -- SKIP (gol)', 'WARN')
        continue

    log(f'[{sf_idx}/{len(subfolders_zona)}] {subfolder.name} -- {tip_str}')
    sf_recv = 0
    sf_sil  = 0

    for f_idx, file_path in enumerate(files, 1):
        fname = file_path.name

        content, tip_detectat, err = read_file_content(str(file_path))
        total_files += 1

        if err or content is None:
            log(f'  EROARE citire [{f_idx}/{len(files)}] {fname}: {err}', 'WARN')
            total_err += 1
            continue

        rows, silents, status = parse_xml_content(content, fname, MAC_RDC_ZONE, mac_to_sector)

        if status != 'OK':
            log(f'  EROARE parsare [{f_idx}/{len(files)}] {fname}: {status}', 'WARN')
            total_err += 1
        else:
            all_rows.extend(rows)
            all_silent.extend(silents)
            sf_recv += len(rows)
            sf_sil  += len(silents)
            if f_idx % 6 == 0 or f_idx == len(files):
                elapsed = (datetime.now() - t_start).seconds
                log(f'  [{f_idx:>3}/{len(files)}] {fname} ({tip_detectat})'                    f'  +{len(rows):>4} recv | +{len(silents):>3} silent'                    f'  [{elapsed}s]')

    log(f'  Subtotal {subfolder.name}: {sf_recv:,} RECEIVED | {sf_sil:,} SILENT')
    total_recv += sf_recv
    total_sil  += sf_sil
    print()

elapsed_total = (datetime.now() - t_start).seconds
print('-' * 60)
log(f'PROCESARE FINALIZATA in {elapsed_total}s', 'INFO')
log(f'Subfoldere procesate: {len(subfolders_zona):>6} din {len(all_dirs)} totale', 'STAT')
log(f'Fisiere dezarhivate : {total_files:>8}', 'STAT')
log(f'Erori               : {total_err:>8}', 'STAT')
log(f'RECEIVED            : {total_recv:>8,}', 'STAT')
log(f'SILENT              : {total_sil:>8,}', 'STAT')
log(f'TOTAL randuri       : {total_recv+total_sil:>8,}', 'STAT')


[08:50:18] [INFO] START procesare Z9 / Luna 10 / 2025
[08:50:18] [INFO] Sursa: /content/drive/MyDrive/RD_03/2025_10_b/2025-10
[08:50:18] [INFO] Gateway-uri zona Z9: ['M70B3D51F1CD8', 'MD83ADD08F4F5', 'MD83ADD08F687']

[08:50:18] [INFO] Total subfoldere (toate zonele): 8
[08:50:18] [INFO] Subfoldere zona Z9: 3 procesate, 5 sarite
[08:50:19] [INFO]   M70B3D51F1CD8        477 x .xml
[08:50:22] [INFO]   MD83ADD08F4F5        745 x .gz
[08:50:22] [INFO]   MD83ADD08F687        745 x .gz

[08:50:22] [INFO] [1/3] M70B3D51F1CD8 -- 477 x .xml
[08:50:33] [INFO]   [  6/477] M70B3D51F1CD8_1759284003_MDP.xml (xml)  + 136 recv | +  0 silent  [11s]
[08:50:34] [INFO]   [ 12/477] M70B3D51F1CD8_1759305602_MDP.xml (xml)  + 156 recv | +  0 silent  [11s]
[08:50:34] [INFO]   [ 18/477] M70B3D51F1CD8_1759327203_MDP.xml (xml)  + 131 recv | +  0 silent  [11s]
[08:50:34] [INFO]   [ 24/477] M70B3D51F1CD8_1759348802_MDP.xml (xml)  + 146 recv | +  0 silent  [12s]
[08:50:34] [INFO]   [ 30/477] M70B3D51F1CD8_1759370402

## Pasul 10 — Construire DataFrame și calcul rx_count_same_ts

In [ ]:
# ════════════════════════════════════════════════════════════════
# PASUL 10 — Construire DataFrame pandas
# + calcul rx_count_same_ts (recepții simultane multi-gateway)
# ════════════════════════════════════════════════════════════════
log('Construire DataFrame principal...', 'INFO')

df_all = pd.DataFrame(all_rows + all_silent)

if df_all.empty:
    log('DataFrame GOL!', 'ERR')
    log('Cauze posibile:', 'WARN')
    log('  1. mac_rdc din lista.txt nu se regăsesc în fișierele .gz', 'WARN')
    log('  2. Folderul sursă nu conține fișiere pentru această zonă', 'WARN')
    log('  3. Verifică Pasul 3 (structura Drive) și Pasul 6 (lista.txt)', 'WARN')
else:
    log(f'DataFrame creat: {len(df_all):,} rânduri × {len(df_all.columns)} coloane', 'OK')

    # Conversie tipuri de date
    log('Conversie tipuri de date...')
    for col in ['rssi','snr','mioty_qi_1','mioty_qi_2','mioty_qi_3',
                'ts','record_start','record_end','bytes','delta_ms']:
        if col in df_all.columns:
            df_all[col] = pd.to_numeric(df_all[col], errors='coerce')

    # rx_count_same_ts: câte gateway-uri diferite au primit
    # același contor + același timestamp + același protocol
    log('Calcul rx_count_same_ts (recepții simultane multiple GW)...')
    mask_recv = df_all['status'] == 'RECEIVED'
    df_r = df_all[mask_recv].copy()
    if not df_r.empty:
        rx = df_r.groupby(
            ['serial_fizic', 'ts', 'protocol']
        )['mac_rdc'].transform('nunique')
        df_all.loc[mask_recv, 'rx_count_same_ts'] = rx.values
    df_all['rx_count_same_ts'] = pd.to_numeric(
        df_all.get('rx_count_same_ts', 1), errors='coerce'
    ).fillna(1).astype(int)

    # ── Statistici distribuție ──
    print()
    log('── Distribuție protocoale (RECEIVED) ──', 'STAT')
    for proto, cnt in df_all[df_all['status']=='RECEIVED']['protocol'].value_counts().items():
        grp = get_group(str(proto))
        log(f'  {str(proto):<8} ({grp:<5}): {cnt:>10,} telegrame', 'STAT')

    print()
    log('── Distribuție status ──', 'STAT')
    for st, cnt in df_all['status'].value_counts().items():
        log(f'  {st:<15}: {cnt:>10,} rânduri', 'STAT')

    print()
    log('── Statistici generale ──', 'STAT')
    log(f'  Seriale fizice unice  : {df_all["serial_fizic"].nunique():>6}', 'STAT')
    log(f'  Gateway-uri (mac_rdc) : {df_all["mac_rdc"].nunique():>6}', 'STAT')
    multi = (df_all['rx_count_same_ts'] > 1).sum()
    log(f'  Recepții multi-GW     : {multi:>6,} rânduri (același ts, >1 GW)', 'STAT')

[08:53:06] [INFO] Construire DataFrame principal...
[08:53:24] [OK] DataFrame creat: 2,018,339 rânduri × 23 coloane
[08:53:24] [INFO] Conversie tipuri de date...
[08:53:32] [INFO] Calcul rx_count_same_ts (recepții simultane multiple GW)...

[08:53:36] [STAT] ── Distribuție protocoale (RECEIVED) ──
[08:53:37] [STAT]   L1C      (L1C  ):    390,197 telegrame
[08:53:37] [STAT]   T1       (T1R3 ):    131,822 telegrame
[08:53:37] [STAT]   R3       (T1R3 ):      3,649 telegrame
[08:53:37] [STAT]   R4P      (R4   ):         15 telegrame

[08:53:37] [STAT] ── Distribuție status ──
[08:53:37] [STAT]   SILENT_SDL     :  1,492,656 rânduri
[08:53:37] [STAT]   RECEIVED       :    525,683 rânduri

[08:53:37] [STAT] ── Statistici generale ──
[08:53:38] [STAT]   Seriale fizice unice  :   3135
[08:53:38] [STAT]   Gateway-uri (mac_rdc) :      3
[08:53:38] [STAT]   Recepții multi-GW     : 94,530 rânduri (același ts, >1 GW)


## Pasul 11 — Salvare fișiere CSV pe protocol

In [ ]:
# ════════════════════════════════════════════════════════════════
# PASUL 11 — Salvare CSV-uri separate pe protocol
#
# Fișiere generate:
#   z{N}_{LUNA}_all.csv    — toate protocoalele + SILENT
#   z{N}_{LUNA}_T1R3.csv   — protocol T1 și R3
#   z{N}_{LUNA}_L1C.csv    — protocol MIOTY L1C
#   z{N}_{LUNA}_R4.csv     — protocoale R4, R4P, R4N
#   z{N}_{LUNA}_OTHER.csv  — alte protocoale nerecunoscute
#   z{N}_{LUNA}_silent.csv — contoare fără recepție în fereastră
# ════════════════════════════════════════════════════════════════
log('Pregătire salvare CSV-uri...', 'INFO')

COL_ORDER = [
    'serial_fizic', 'device_id_raw', 'decode_fmt',
    'mac_rdc', 'sector_id', 'zona',
    'protocol', 'protocol_group',
    'ts', 'ts_human', 'delta_ms', 'bytes',
    'rssi', 'snr',
    'mioty_qi_1', 'mioty_qi_2', 'mioty_qi_3',
    'aID',
    'record_start', 'record_end',
    'rx_count_same_ts', 'status', 'gz_source'
]

# Asigură că toate coloanele există
for col in COL_ORDER:
    if col not in df_all.columns:
        df_all[col] = None
df_out = df_all[COL_ORDER].copy()

saved = {}

def save_csv(df_sub, suffix, label):
    path = f'{OUTPUT_DIR}/z{ZONA_NR}_{LUNA}_{suffix}.csv'
    df_sub.to_csv(path, index=False, encoding='utf-8-sig')
    size_kb = os.path.getsize(path) // 1024
    log(f'z{ZONA_NR}_{LUNA}_{suffix}.csv  —  {len(df_sub):,} rânduri  —  {size_kb:,} KB', 'SAVE')
    saved[label] = path
    return path

# Salvare fișiere
save_csv(df_out, 'all', 'ALL')
save_csv(df_out[df_out['protocol_group']=='T1R3'], 'T1R3',  'T1R3')
save_csv(df_out[df_out['protocol_group']=='L1C'],  'L1C',   'L1C')
save_csv(df_out[df_out['protocol_group']=='R4'],   'R4',    'R4')

df_other = df_out[
    df_out['protocol_group'].isin(['OTHER']) |
    (df_out['protocol_group'].isna() & (df_out['status']=='RECEIVED'))
]
if len(df_other) > 0:
    save_csv(df_other, 'OTHER', 'OTHER')

df_sil = df_out[df_out['status'].isin(['SILENT','SILENT_SDL'])]
if len(df_sil) > 0:
    save_csv(df_sil, 'silent', 'SILENT')

print()
log('Toate CSV-urile salvate cu succes!', 'OK')

[08:53:38] [INFO] Pregătire salvare CSV-uri...
[08:54:10] [SAVE] z9_10_all.csv  —  2,018,339 rânduri  —  266,637 KB
[08:54:12] [SAVE] z9_10_T1R3.csv  —  135,471 rânduri  —  21,736 KB
[08:54:21] [SAVE] z9_10_L1C.csv  —  390,197 rânduri  —  64,921 KB
[08:54:21] [SAVE] z9_10_R4.csv  —  15 rânduri  —  2 KB
[08:54:41] [SAVE] z9_10_silent.csv  —  1,492,656 rânduri  —  179,978 KB

[08:54:41] [OK] Toate CSV-urile salvate cu succes!


## Pasul 12 — Calcul rată de citire lunară (TSR)

In [ ]:
# ════════════════════════════════════════════════════════════════
# PASUL 12 — Telegram Success Rate (TSR) per contor-GW-protocol
# Aceasta este baza pentru analiza ratei de citire lunare
#
# TSR = n_telegrame_primite / n_telegrame_asteptate
# Clasificare: GREEN ≥ 0.90 | YELLOW 0.70-0.90 | RED ≤ 0.70
# ════════════════════════════════════════════════════════════════
log('Calcul TSR (Telegram Success Rate) per contor-GW-protocol...', 'INFO')

df_recv = df_out[df_out['status'] == 'RECEIVED'].copy()
tsr_rows = []

if df_recv.empty:
    log('Nicio telegramă RECEIVED — TSR nu poate fi calculat.', 'WARN')
else:
    grp_count = df_recv.groupby(['serial_fizic','mac_rdc','protocol']).ngroups
    log(f'Calculez TSR pentru {grp_count:,} combinații contor-GW-protocol...')

    for (serial, mac, proto), grp in df_recv.groupby(['serial_fizic','mac_rdc','protocol']):
        ts_vals    = grp['ts'].dropna().sort_values().astype(int)
        if len(ts_vals) < 2:
            continue

        durata     = int(ts_vals.max() - ts_vals.min())
        cadenta    = CADENTA.get(str(proto).upper(), 3600)
        n_asteptat = max(1, durata // cadenta)
        n_primit   = len(ts_vals)
        tsr        = min(1.0, round(n_primit / n_asteptat, 4))

        deltas     = ts_vals.diff().dropna()
        gap_big    = int((deltas > 60).sum())
        delta_irr  = round(gap_big / max(1, len(deltas)), 4)

        # Statistici radio
        rssi_s = grp['rssi'].dropna()
        snr_s  = grp['snr'].dropna()

        flag = 'GREEN' if tsr >= 0.90 else ('RED' if tsr <= 0.70 else 'YELLOW')

        tsr_rows.append({
            'serial_fizic'      : serial,
            'mac_rdc'           : mac,
            'sector_id'         : grp['sector_id'].iloc[0],
            'zona'              : ZONA,
            'protocol'          : proto,
            'protocol_group'    : get_group(proto),
            'luna'              : LUNA,
            'an'                : AN,
            'n_telegrame'       : n_primit,
            'n_asteptat'        : int(n_asteptat),
            'TSR'               : tsr,
            'ts_min'            : int(ts_vals.min()),
            'ts_max'            : int(ts_vals.max()),
            'durata_ore'        : round(durata / 3600, 2),
            'rssi_mean'         : round(float(rssi_s.mean()), 2)    if len(rssi_s)>0  else None,
            'rssi_stddev'       : round(float(rssi_s.std()), 2)     if len(rssi_s)>1  else None,
            'rssi_variation'    : round(float(rssi_s.max()-rssi_s.min()),2) if len(rssi_s)>1 else None,
            'snr_mean'          : round(float(snr_s.mean()), 2)     if len(snr_s)>0   else None,
            'snr_stddev'        : round(float(snr_s.std()), 2)      if len(snr_s)>1   else None,
            'median_delta_t'    : round(float(deltas.median()), 2)  if len(deltas)>0  else None,
            'gap_count_big'     : gap_big,
            'delta_irregularity': delta_irr,
            'radio_quality_flag': flag,
        })

    df_tsr = pd.DataFrame(tsr_rows)
    path_tsr = f'{OUTPUT_DIR}/z{ZONA_NR}_{LUNA}_rata_citire.csv'
    df_tsr.to_csv(path_tsr, index=False, encoding='utf-8-sig')
    size_kb = os.path.getsize(path_tsr) // 1024
    log(f'z{ZONA_NR}_{LUNA}_rata_citire.csv  —  {len(df_tsr):,} linkuri  —  {size_kb:,} KB', 'SAVE')

    print()
    log('── Sumar rată de citire ──', 'STAT')
    for flag, culoare in [('GREEN','GREEN'),('YELLOW','YELLOW'),('RED','RED')]:
        cnt  = (df_tsr['radio_quality_flag'] == flag).sum()
        pct  = cnt / max(1, len(df_tsr)) * 100
        log(f'  {culoare} {flag:<8}: {cnt:>5} linkuri ({pct:.1f}%)', 'STAT')
    log(f'  TSR mediu general: {df_tsr["TSR"].mean():.4f}', 'STAT')

[08:54:41] [INFO] Calcul TSR (Telegram Success Rate) per contor-GW-protocol...
[08:54:41] [INFO] Calculez TSR pentru 3,582 combinații contor-GW-protocol...
[08:54:49] [SAVE] z9_10_rata_citire.csv  —  3,427 linkuri  —  468 KB

[08:54:49] [STAT] ── Sumar rată de citire ──
[08:54:49] [STAT]   GREEN GREEN   :    18 linkuri (0.5%)
[08:54:49] [STAT]   YELLOW YELLOW  :     1 linkuri (0.0%)
[08:54:49] [STAT]   RED RED     :  3408 linkuri (99.4%)
[08:54:49] [STAT]   TSR mediu general: 0.1797


## Pasul 13 —  Raport final și lista fișierelor generate

In [ ]:
# ================================================================
# PASUL 13 -- Raport final
# ================================================================
print()
print('=' * 62)
print('  RAPORT FINAL PROCESARE')
print('=' * 62)
print(f'  Zona         : {ZONA}')
print(f'  Luna         : {LUNA}')
print(f'  An           : {AN}')
print(f'  Sursa        : {SURSA_BASE}')
print('-' * 62)
print(f'  Subfoldere zona procesate : {len(subfolders_zona)} din {len(all_dirs)} totale')
print(f'  Fisiere dezarhivate       : {total_files}')
print(f'  Erori parsare             : {total_err}')
print(f'  Telegrame RECEIVED        : {total_recv:,}')
print(f'  Contoare SILENT           : {total_sil:,}')
n_unice = df_all["serial_fizic"].nunique() if not df_all.empty else 0
print(f'  Seriale fizice unice      : {n_unice:,}')
print('-' * 62)
print('  FISIERE GENERATE IN DRIVE:')
for label, path in saved.items():
    fname = Path(path).name
    size  = os.path.getsize(path)//1024 if os.path.exists(path) else 0
    print(f'    {fname:<44} {size:>5} KB')
path_tsr_check = f'{OUTPUT_DIR}/z{ZONA_NR}_{LUNA}_rata_citire.csv'
if os.path.exists(path_tsr_check):
    fname = Path(path_tsr_check).name
    size  = os.path.getsize(path_tsr_check)//1024
    print(f'    {fname:<44} {size:>5} KB')
print(f'  Output folder: {OUTPUT_DIR}')
print('=' * 62)
print()
log('PROCESARE COMPLETA CU SUCCES!', 'OK')

# -- Scriere sumar final in log --
with open(LOG_PATH, 'a', encoding='utf-8') as lf:
    lf.write('-' * 60 + '\n')
    lf.write('SUMAR FINAL\n')
    lf.write(f'  Subfoldere zona   : {len(subfolders_zona)} din {len(all_dirs)} totale\n')
    lf.write(f'  Fisiere procesate : {total_files}\n')
    lf.write(f'  Erori             : {total_err}\n')
    lf.write(f'  RECEIVED          : {total_recv:,}\n')
    lf.write(f'  SILENT            : {total_sil:,}\n')
    lf.write(f'  Seriale unice     : {n_unice:,}\n')
    lf.write('  Fisiere generate:\n')
    for label, path in saved.items():
        fname = Path(path).name
        size  = os.path.getsize(path)//1024 if os.path.exists(path) else 0
        lf.write(f'    {fname}  ({size} KB)\n')
    lf.write(f'End: {datetime.now().strftime("%Y-%m-%d %H:%M:%S")}\n')
    lf.write('-' * 60 + '\n')
log(f'Log salvat in: {LOG_PATH}', 'INFO')



  RAPORT FINAL PROCESARE
  Zona         : Z9
  Luna         : 10
  An           : 2025
  Sursa        : /content/drive/MyDrive/RD_03/2025_10_b/2025-10
--------------------------------------------------------------
  Subfoldere zona procesate : 3 din 8 totale
  Fisiere dezarhivate       : 1967
  Erori parsare             : 0
  Telegrame RECEIVED        : 525,683
  Contoare SILENT           : 1,492,656
  Seriale fizice unice      : 3,135
--------------------------------------------------------------
  FISIERE GENERATE IN DRIVE:
    z9_10_all.csv                                266637 KB
    z9_10_T1R3.csv                               21736 KB
    z9_10_L1C.csv                                64921 KB
    z9_10_R4.csv                                     2 KB
    z9_10_silent.csv                             179978 KB
    z9_10_rata_citire.csv                          468 KB
  Output folder: /content/drive/MyDrive/deva/10/Z9

[08:54:49] [OK] PROCESARE COMPLETA CU SUCCES!
[08:54:49] [INFO] L

## Pasul 14 —  Preview date (opțional, verificare vizuală)

In [ ]:
# ════════════════════════════════════════════════════════════════
# PASUL 14 — Preview primele rânduri pentru fiecare grup de protocol
# Opțional — rulează dacă vrei să verifici datele vizual
# ════════════════════════════════════════════════════════════════
if df_out.empty:
    print('Nu există date de afișat.')
else:
    cols_p = ['serial_fizic','mac_rdc','sector_id','protocol',
              'ts_human','rssi','snr','bytes','rx_count_same_ts','status']
    cols_p = [c for c in cols_p if c in df_out.columns]

    for grp_name in ['T1R3','L1C','R4']:
        sub = df_out[(df_out['protocol_group']==grp_name) & (df_out['status']=='RECEIVED')]
        if not sub.empty:
            print(f'\n━━━ {grp_name}  —  {len(sub):,} rânduri totale  ━━━')
            display(sub[cols_p].head(5))

    sil = df_out[df_out['status'].isin(['SILENT','SILENT_SDL'])]
    if not sil.empty:
        print(f'\n━━━ SILENT  —  {len(sil):,} rânduri totale  ━━━')
        display(sil[cols_p].head(5))

    # Preview rata de citire
    if 'df_tsr' in dir() and not df_tsr.empty:
        print(f'\n━━━ RATA DE CITIRE (TSR)  —  primele 10 rânduri  ━━━')
        display(df_tsr.head(10))


━━━ T1R3  —  135,471 rânduri totale  ━━━


,serial_fizic,mac_rdc,sector_id,protocol,ts_human,rssi,snr,bytes,rx_count_same_ts,status
18,68478118,M70B3D51F1CD8,ABBY0001b,R3,2025-09-30 21:00:00,-95.0,NaN,63,1,RECEIVED
21,68478118,M70B3D51F1CD8,ABBY0001b,R3,2025-09-30 20:46:38,-95.0,NaN,63,1,RECEIVED
53,68478145,M70B3D51F1CD8,ABBY0001b,R3,2025-09-30 20:59:50,-95.0,NaN,63,1,RECEIVED
54,68478145,M70B3D51F1CD8,ABBY0001b,R3,2025-09-30 20:54:56,-95.0,NaN,63,1,RECEIVED
87,78282863,M70B3D51F1CD8,ABBY0001b,R3,2025-09-30 20:59:22,-101.0,NaN,63,1,RECEIVED



━━━ L1C  —  390,197 rânduri totale  ━━━


,serial_fizic,mac_rdc,sector_id,protocol,ts_human,rssi,snr,bytes,rx_count_same_ts,status
0,78281899,M70B3D51F1CD8,ABBY0001b,L1C,2025-09-30 20:52:44,NaN,-21.0,79,1,RECEIVED
1,68783279,M70B3D51F1CD8,ABBY0001b,L1C,2025-09-30 20:02:53,NaN,-21.0,79,1,RECEIVED
2,78281849,M70B3D51F1CD8,ABBY0001b,L1C,2025-09-30 20:44:31,NaN,-21.0,79,1,RECEIVED
3,18862549,M70B3D51F1CD8,ABBY0001b,L1C,2025-09-30 20:25:14,NaN,-21.0,79,1,RECEIVED
4,18861549,M70B3D51F1CD8,ABBY0001b,L1C,2025-09-30 20:12:27,NaN,-21.0,79,1,RECEIVED



━━━ R4  —  15 rânduri totale  ━━━


,serial_fizic,mac_rdc,sector_id,protocol,ts_human,rssi,snr,bytes,rx_count_same_ts,status
35242,68378801,M70B3D51F1CD8,ABBY0001b,R4P,2025-10-12 04:06:04,NaN,-13.0,84,1,RECEIVED
60710,68783237,M70B3D51F1CD8,ABBY0001b,R4P,2025-10-30 19:39:23,NaN,-10.0,84,1,RECEIVED
170165,88226605,MD83ADD08F4F5,ABCB0001,R4P,2025-10-11 07:04:18,NaN,-3.0,84,1,RECEIVED
287617,68083318,MD83ADD08F4F5,ABCB0001,R4P,2025-10-22 11:06:07,NaN,-9.0,84,2,RECEIVED
307414,68691018,MD83ADD08F4F5,ABCB0001,R4P,2025-10-24 06:59:24,NaN,6.0,84,1,RECEIVED



━━━ SILENT  —  1,492,656 rânduri totale  ━━━


,serial_fizic,mac_rdc,sector_id,protocol,ts_human,rssi,snr,bytes,rx_count_same_ts,status
525683,81325001,MD83ADD08F4F5,ABCB0001,None,,NaN,NaN,0,1,SILENT_SDL
525684,81325008,MD83ADD08F4F5,ABCB0001,None,,NaN,NaN,0,1,SILENT_SDL
525685,81325009,MD83ADD08F4F5,ABCB0001,None,,NaN,NaN,0,1,SILENT_SDL
525686,81542227,MD83ADD08F4F5,ABCB0001,None,,NaN,NaN,0,1,SILENT_SDL
525687,81682014,MD83ADD08F4F5,ABCB0001,None,,NaN,NaN,0,1,SILENT_SDL



━━━ RATA DE CITIRE (TSR)  —  primele 10 rânduri  ━━━


,serial_fizic,mac_rdc,sector_id,zona,protocol,protocol_group,luna,an,n_telegrame,n_asteptat,...,durata_ore,rssi_mean,rssi_stddev,rssi_variation,snr_mean,snr_stddev,median_delta_t,gap_count_big,delta_irregularity,radio_quality_flag
0,18158916,MD83ADD08F687,ABBY0001a,Z9,L1C,L1C,10,2025,20,684,...,684.92,-137.65,0.88,4.0,-0.75,0.55,11312.0,19,1.0000,RED
1,18185217,M70B3D51F1CD8,ABBY0001b,Z9,L1C,L1C,10,2025,179,741,...,741.08,NaN,NaN,NaN,-21.00,0.00,10971.0,158,0.8876,RED
2,18185217,MD83ADD08F4F5,ABCB0001,Z9,L1C,L1C,10,2025,277,741,...,741.08,-122.18,1.38,11.0,13.75,1.40,10971.0,246,0.8913,RED
3,18185217,MD83ADD08F687,ABBY0001a,Z9,L1C,L1C,10,2025,90,741,...,741.08,-132.89,1.11,5.0,2.98,1.07,10971.0,79,0.8876,RED
4,18186200,M70B3D51F1CD8,ABBY0001b,Z9,L1C,L1C,10,2025,175,743,...,743.98,NaN,NaN,NaN,-21.00,0.00,10341.0,155,0.8908,RED
5,18186200,MD83ADD08F4F5,ABCB0001,Z9,L1C,L1C,10,2025,276,743,...,743.99,-106.22,4.90,34.0,25.04,3.20,10341.0,246,0.8945,RED
6,18186200,MD83ADD08F4F5,ABCB0001,Z9,T1,T1R3,10,2025,726,166915,...,741.85,-92.29,3.01,18.0,NaN,NaN,3600.0,725,1.0000,RED
7,18186200,MD83ADD08F687,ABBY0001a,Z9,L1C,L1C,10,2025,18,684,...,684.11,-136.67,0.97,4.0,-0.22,0.73,10701.0,15,0.8824,RED
8,18186225,MD83ADD08F4F5,ABCB0001,Z9,L1C,L1C,10,2025,254,737,...,737.90,-133.46,2.30,10.0,2.83,2.20,10773.0,225,0.8893,RED
9,18186238,MD83ADD08F4F5,ABCB0001,Z9,L1C,L1C,10,2025,46,728,...,728.69,-136.83,1.50,5.0,0.11,1.08,11547.0,43,0.9556,RED
